# La discrepancia: mismo código, dos resultados
### Punto de partida del caso

El informe de ingresos del punto de venta piloto de Aroma Andino se calculaba con un análisis ad hoc: un notebook sin estructura que leía el
extracto del POS y cargaba a una base local. El informe del lunes reportó \$606.000; el del martes, \$1.212.000; no hubo cambios en el código entre ambas ejecuciones. Las cifras no conciliaban y ninguna alerta lo señaló.

Este notebook reproduce el antipatrón tal como ocurrió. Funciona sobre una copia recién descargada del repositorio. Las dos celdas de carga son idénticas y representan la misma celda ejecutada dos veces. La causa raíz es una carga no idempotente (`if_exists="append"`); una operación es idempotente cuando ejecutarla muchas veces deja el sistema exactamente igual que ejecutarla una vez.

In [1]:
# Preparación: garantizar el snapshot de fuentes (solo la capa raw)
# En una copia recién descargada, esta celda escribe data/raw si no existe.
# No ejecuta el pipeline ni ninguna validación: solo deja las fuentes listas.
import os, sys; os.chdir(".."); sys.path.insert(0, os.getcwd())
from src.config import leer_config
from src.respaldo_fuentes import generar_fuentes

cfg = leer_config()
rutas = generar_fuentes(cfg)
for nombre, ruta in rutas.items():
    print(f"{nombre:10s} -> {ruta.relative_to(ruta.parents[2])}")

ventas     -> data\raw\ventas_api.json
clientes   -> data\raw\clientes.html
productos  -> data\raw\productos.csv


In [2]:
# Lectura rápida del extracto del POS y del catálogo del ERP
import json, sqlite3, pathlib
import pandas as pd

raw = pathlib.Path("data/raw")
ventas = pd.DataFrame(json.loads((raw / "ventas_api.json").read_text(encoding="utf-8"))["data"])
productos = pd.read_csv(raw / "productos.csv")

df = ventas.merge(productos[["producto_id", "precio_unitario"]], on="producto_id")
df = df[df["cantidad"] > 0]                       # descarta anulaciones y reversas del POS
df["ingreso"] = df["cantidad"] * df["precio_unitario"]

con = sqlite3.connect("discrepancia/discrepancia.db")
con.execute("DROP TABLE IF EXISTS ventas")        # estado inicial limpio
con.commit()
len(df)

412

In [3]:
# Ejecución #1 de la celda de carga (informe del lunes)
df.to_sql("ventas", con, if_exists="append", index=False)
kpi = pd.read_sql("SELECT COUNT(*) AS filas, SUM(ingreso) AS ingreso_total FROM ventas", con)
kpi

,filas,ingreso_total
0,412,606000


In [4]:
# Ejecución #2 de la MISMA celda de carga (informe del martes)
df.to_sql("ventas", con, if_exists="append", index=False)
kpi = pd.read_sql("SELECT COUNT(*) AS filas, SUM(ingreso) AS ingreso_total FROM ventas", con)
kpi

,filas,ingreso_total
0,824,1212000


## ¿Qué ocurrió?

La carga con `append` no es idempotente, ya que la segunda ejecución duplicó los 412 tiquetes (824) y el ingreso reportado pasó de \$606.000 a \$1.212.000. Entonces, la discrepancia entre informes se dió porque indicador cambió.

El pipeline del repositorio previene la recurrencia. Es decir, la regla de unicidad detiene la publicación ante `venta_id` duplicados y la carga usa reemplazo idempotente (`if_exists="replace"`). Los duplicados no solo nacen de un append mal usado, pues también podrían venir de la propia fuente; por ejemplo, que la API del POS envíe el mismo tiquete dos veces o que alguien integre dos extractos que se superponen.

In [5]:
# El pipeline del repositorio: mismo indicador, proceso reproducible
con.close()
from src.pipeline import run_pipeline

r1 = run_pipeline(cfg)
r2 = run_pipeline(cfg)   # segunda ejecución completa

print(f"ejecución 1 -> {r1['filas']} filas · $ {r1['ingreso_total']:,} · hash {r1['hash'][:12]}…")
print(f"ejecución 2 -> {r2['filas']} filas · $ {r2['ingreso_total']:,} · hash {r2['hash'][:12]}…")
print("hashes idénticos:", r1["hash"] == r2["hash"])

ejecución 1 -> 412 filas · $ 606,000 · hash 47f91b7be24f…
ejecución 2 -> 412 filas · $ 606,000 · hash 47f91b7be24f…
hashes idénticos: True
